# Project 2: Unemployment Analysis with Python

Beginner project. Goal: explore unemployment measurements over time, make charts, and describe patterns without making unsupported cause-and-effect claims.

## 1. Load the data
The notebook uses a public CSV URL. If it stops working, upload a suitable unemployment CSV to Colab and replace the URL with the uploaded filename.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
url = 'https://raw.githubusercontent.com/datasets/unemployment/master/data/unemployment.csv'
df = pd.read_csv(url)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(10))
df.info()

## 2. Check missing values and find columns
Column names differ between datasets. Check the output and adjust the selected columns in the next cell if needed.

In [ ]:
print('Missing values:\n', df.isna().sum())
date_candidates = [c for c in df.columns if c.lower() in ['date','month','time'] or 'date' in c.lower()]
value_candidates = [c for c in df.columns if any(k in c.lower() for k in ['unemployment','rate','value'])]
print('Possible date columns:', date_candidates)
print('Possible rate/value columns:', value_candidates)

## 3. Clean dates and values
This cell guesses common column names. If the guesses are wrong, change `date_col` and `rate_col` to the correct names printed above.

In [ ]:
date_col = date_candidates[0] if date_candidates else df.columns[0]
rate_col = value_candidates[0] if value_candidates else df.columns[-1]
print('Using:', date_col, 'and', rate_col)
work = df[[date_col, rate_col]].copy()
work.columns = ['date','unemployment_rate']
work['date'] = pd.to_datetime(work['date'], errors='coerce')
work['unemployment_rate'] = pd.to_numeric(work['unemployment_rate'], errors='coerce')
work = work.dropna(subset=['date','unemployment_rate']).sort_values('date')
print('Rows after cleaning:', len(work))
display(work.head())

## 4. Plot the trend
A line chart shows how the chosen measure changes over time. Confirm the source units before interpreting the y-axis.

In [ ]:
plt.figure(figsize=(12,5))
sns.lineplot(data=work, x='date', y='unemployment_rate')
plt.title('Unemployment measure over time'); plt.xlabel('Date'); plt.ylabel('Rate/value (check units)')
plt.tight_layout(); plt.show()

## 5. Summary and annual averages
The average describes a typical value in the selected data. Annual averages can hide changes within each year.

In [ ]:
print(work['unemployment_rate'].describe())
print('Date range:', work['date'].min(), 'to', work['date'].max())
work['year'] = work['date'].dt.year
yearly = work.groupby('year', as_index=False)['unemployment_rate'].mean()
display(yearly.tail(15))
plt.figure(figsize=(12,5))
sns.barplot(data=yearly, x='year', y='unemployment_rate', color='steelblue')
plt.title('Average measure by year'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 6. Compare 2019 and 2020 carefully
This is a simple comparison, not proof that COVID-19 caused a change. Check that the data covers both years and the same geography/definition.

In [ ]:
period = yearly[yearly['year'].isin([2019, 2020])]
display(period)
if set(period['year']) == {2019, 2020}:
    vals = period.set_index('year')['unemployment_rate']
    print('2020 minus 2019:', round(vals.loc[2020]-vals.loc[2019], 3), 'dataset units')
else:
    print('Both 2019 and 2020 are not present in this dataset.')

## 7. Conclusion and improvement
Write the geography, units, date range, overall trend, highest/lowest years, and limitations. A trend alone does not prove why something happened. **Extra idea:** for regular monthly data with several years, add a rolling 12-month average; if several regions exist, compare them only when definitions match.

**GitHub:** save as `Unemployment_Analysis.ipynb`, document the data source and limitations, and use repository name `CodeAlpha_Unemployment_Analysis`. Run all cells and verify the charts before submitting.